# Processing Layer (Analysis Layer)
## Big Data Analytics Project - Mobile Device Market Analysis

This notebook covers the **Processing/Analysis Layer** of our Big Data Analytics project.

### Objectives:
- Process data using Spark and MapReduce
- Feature extraction for Machine Learning
- Train Machine Learning models
- Generate added value from data
- Apply data analytics methods

In [1]:
import syssys.path.append('..')import pandas as pdimport numpy as npimport warningswarnings.filterwarnings('ignore')# Spark importsfrom pyspark.sql import SparkSessionfrom pyspark.sql.functions import col, avg, count, max as spark_max, min as spark_min, when# ML importsfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import mean_squared_error, r2_score, accuracy_score# Project modulesfrom src.spark_processing import (    create_spark_session,    pandas_to_spark_df,    spark_mapreduce_example,    process_device_data,    spark_ml_preprocessing)from src.ml_models import (    prepare_features,    train_price_prediction_model,    train_category_classification_model)print("SUCCESS: All imports successful!")

SyntaxError: invalid syntax (538394305.py, line 1)

## 1. Load Data from Storage Layer

In [ ]:
# Load stored data
df = pd.read_csv("../data/stored_data.csv")

print(f"Loaded {len(df)} records for processing")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Spark and MapReduce Processing

Process data using Apache Spark and MapReduce patterns.

In [ ]:
# Create Spark session
spark = create_spark_session("MobileDeviceAnalytics")

print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")

### Convert to Spark DataFrame

In [ ]:
# Convert pandas DataFrame to Spark DataFrame
spark_df = pandas_to_spark_df(spark, df)

print("Spark DataFrame created")
print(f"Number of partitions: {spark_df.rdd.getNumPartitions()}")
spark_df.show(5)

### MapReduce Example

MapReduce pattern: Group by category (Map) and aggregate statistics (Reduce)

In [ ]:
# Add category column if not exists
if 'category' not in df.columns and 'price' in df.columns:
    df['category'] = pd.cut(df['price'], 
                           bins=[0, 500, 1000, float('inf')],
                           labels=['Budget', 'Mid-range', 'Premium'])
    spark_df = pandas_to_spark_df(spark, df)

# Perform MapReduce operation
if 'category' in [col.name for col in spark_df.schema.fields]:
    mapreduce_result = spark_mapreduce_example(spark, spark_df)
    print("\nMapReduce Results (Group by Category):")
    mapreduce_result.show()

### Spark Data Transformations

In [ ]:
# Process device data with Spark
processed_spark_df = process_device_data(spark, df)

print("Processed Spark DataFrame:")
processed_spark_df.select("device_name", "brand", "price", "price_category", "year").show(10)

## 3. Feature Extraction for Machine Learning

Extract and engineer features for ML models.

In [ ]:
# Prepare features
features_df, le_brand, le_os = prepare_features(df)

print("Feature Engineering Complete")
print(f"\nNew features created:")
print(f"  - brand_encoded: {('brand_encoded' in features_df.columns)}")
print(f"  - os_encoded: {('os_encoded' in features_df.columns)}")
if 'ram_storage_ratio' in features_df.columns:
    print(f"  - ram_storage_ratio: Created")
if 'price_per_inch' in features_df.columns:
    print(f"  - price_per_inch: Created")

print(f"\nFeature DataFrame shape: {features_df.shape}")
features_df.head()

## 4. Machine Learning ModelsTrain ML models to generate added value from data.

### 4.1 Price Prediction Model (Regression)

In [ ]:
# Train price prediction model
if 'price' in df.columns:
    price_model = train_price_prediction_model(df)
    
    print("Price Prediction Model Results:")
    print(f"  RMSE: ${price_model['metrics']['rmse']:.2f}")
    print(f"  R² Score: {price_model['metrics']['r2_score']:.4f}")
    
    print("\nFeature Importance:")
    print(price_model['feature_importance'])
else:
    print("WARNING: Price column not found in dataset")

### 4.2 Category Classification Model

In [ ]:
# Train category classification model
category_model = train_category_classification_model(df)

print("Category Classification Model Results:")
    print(f"  Accuracy: {category_model['metrics']['accuracy']:.4f}")
    print("\nClassification Report:")
print(category_model['metrics']['classification_report'])

## 5. Save Processed Data and Models

In [ ]:
# Save processed data
processed_df = features_df.copy()
output_path = "../data/processed_data.csv"
processed_df.to_csv(output_path, index=False)
print(f"SUCCESS: Processed data saved to {output_path}")

# Save models
if 'price' in df.columns:
    from ml_models import save_model
    save_model(price_model, "../data/price_model.pkl")
    save_model(category_model, "../data/category_model.pkl")

# Stop Spark session
spark.stop()
print("Spark session stopped")

## Summary

**Processing Layer Complete**

- Processed data using Spark
- Implemented MapReduce pattern
- Extracted features for ML
- Trained price prediction model (Regression)
- Trained category classification model
- Generated added value from data
